# 📘 Module 1.3 — Your First Neural Network

**Goal:** Build, train, and evaluate a neural network using `torch.nn` on a real dataset.

## Why This Matters for ADAS
Every deep learning model in ADAS — from object detectors to trajectory predictors — follows the same pattern:
1. Define a model (layers)
2. Define a loss function
3. Train with an optimizer
4. Evaluate on test data

We'll learn this pattern with a simple example, then use it for everything else.

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load the MNIST Dataset

MNIST contains 70,000 handwritten digits (0-9). Each image is 28×28 pixels, grayscale.

**Why MNIST?** It's the "Hello World" of deep learning — simple enough to train quickly, complex enough to show real patterns.

In [ ]:
# --- Data Loading Pipeline ---
# This pattern is used for ALL datasets in PyTorch

# Transform: convert images to tensors and normalize
transform = transforms.Compose([
    transforms.ToTensor(),           # PIL Image → Tensor, scales to [0, 1]
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1]
])

# Download and load datasets
train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform
)

# DataLoader: handles batching, shuffling, parallel loading
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Number of classes: {len(train_dataset.classes)}")
print(f"Image shape: {train_dataset[0][0].shape}")

In [ ]:
# --- Visualize some samples ---
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}')
    ax.axis('off')
plt.suptitle('MNIST Samples', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Define the Neural Network

We'll build a simple **Multi-Layer Perceptron (MLP)** — a stack of fully-connected layers.

```
Input (784) → Linear(256) → ReLU → Linear(128) → ReLU → Linear(10) → Output
```

**Architecture explained:**
- **Input:** 28×28 = 784 pixels (flattened)
- **Hidden layers:** Learn increasingly abstract features
- **Output:** 10 neurons (one per digit class)

In [ ]:
class SimpleNN(nn.Module):
    """
    A simple feedforward neural network.
    All PyTorch models inherit from nn.Module.
    """
    def __init__(self):
        super(SimpleNN, self).__init__()
        
        # Define layers
        self.flatten = nn.Flatten()            # 28x28 → 784
        self.fc1 = nn.Linear(784, 256)         # Fully connected layer 1
        self.fc2 = nn.Linear(256, 128)         # Fully connected layer 2
        self.fc3 = nn.Linear(128, 10)          # Output layer (10 classes)
        self.relu = nn.ReLU()                  # Activation function
        self.dropout = nn.Dropout(0.2)         # Regularization
    
    def forward(self, x):
        """Forward pass — defines how data flows through the network."""
        x = self.flatten(x)        # Flatten image
        x = self.relu(self.fc1(x)) # Layer 1 + activation
        x = self.dropout(x)        # Dropout for regularization
        x = self.relu(self.fc2(x)) # Layer 2 + activation
        x = self.dropout(x)
        x = self.fc3(x)            # Output (raw logits, no softmax)
        return x

# Create model
model = SimpleNN().to(device)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 3. Loss Function & Optimizer

- **Loss function:** Measures how wrong our predictions are
- **Optimizer:** Updates model parameters to minimize the loss

In [ ]:
# CrossEntropyLoss = Softmax + NLL Loss (standard for classification)
criterion = nn.CrossEntropyLoss()

# Adam optimizer (adaptive learning rate — works well in most cases)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

## 4. Training Loop ⚡

The universal pattern for ALL PyTorch training:
```
for each epoch:
    for each batch:
        1. Forward pass (model prediction)
        2. Compute loss
        3. Backward pass (compute gradients)
        4. Update weights (optimizer step)
        5. Zero gradients
```

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """Train the model for one epoch."""
    model.train()  # Set to training mode (enables dropout)
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        # Move data to device
        images, labels = images.to(device), labels.to(device)
        
        # 1. Forward pass
        outputs = model(images)
        
        # 2. Compute loss
        loss = criterion(outputs, labels)
        
        # 3. Backward pass
        optimizer.zero_grad()  # Zero gradients from previous step
        loss.backward()        # Compute gradients
        
        # 4. Update weights
        optimizer.step()
        
        # Track metrics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def evaluate(model, test_loader, criterion, device):
    """Evaluate the model on test data."""
    model.eval()  # Set to evaluation mode (disables dropout)
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # No gradients needed for evaluation
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    test_loss = running_loss / len(test_loader)
    test_acc = 100. * correct / total
    return test_loss, test_acc

In [ ]:
# --- Train the model ---
NUM_EPOCHS = 5

train_losses, test_losses = [], []
train_accs, test_accs = [], []

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

In [ ]:
# --- Plot training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label='Train Loss')
ax1.plot(test_losses, label='Test Loss')
ax1.set_title('Loss Curves')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_accs, label='Train Accuracy')
ax2.plot(test_accs, label='Test Accuracy')
ax2.set_title('Accuracy Curves')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Visualize Predictions

In [ ]:
# --- Show predictions on test images ---
model.eval()
fig, axes = plt.subplots(2, 8, figsize=(16, 4))

with torch.no_grad():
    for i, ax in enumerate(axes.flat):
        image, true_label = test_dataset[i]
        output = model(image.unsqueeze(0).to(device))
        pred_label = output.argmax(dim=1).item()
        confidence = torch.softmax(output, dim=1).max().item()
        
        ax.imshow(image.squeeze(), cmap='gray')
        color = 'green' if pred_label == true_label else 'red'
        ax.set_title(f'P:{pred_label} ({confidence:.0%})', color=color, fontsize=10)
        ax.axis('off')

plt.suptitle('Model Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Save & Load Model

In [ ]:
# --- Save model weights ---
# This is how you save and share trained models
# torch.save(model.state_dict(), 'mnist_model.pth')
# print("Model saved!")

# --- Load model weights ---
# loaded_model = SimpleNN().to(device)
# loaded_model.load_state_dict(torch.load('mnist_model.pth'))
# loaded_model.eval()
# print("Model loaded!")

print("💡 Uncomment the above code to save/load models")
print("In ADAS, trained model weights are deployed to edge devices (e.g., NVIDIA Jetson)")

---
## ✅ Key Takeaways

1. **All PyTorch models** inherit from `nn.Module` and implement `forward()`
2. **DataLoader** handles batching, shuffling, and parallel data loading
3. **Training loop:** Forward → Loss → Backward → Step → Zero Grad
4. **`model.train()`** enables dropout/batchnorm; **`model.eval()`** disables them
5. **`torch.no_grad()`** disables gradient tracking during evaluation (saves memory)

## 🧪 Exercises
1. Change the hidden layer sizes. How does it affect accuracy?
2. Try different optimizers: `SGD`, `RMSprop`. Compare convergence.
3. Add more hidden layers. Does deeper = better?
4. Try without dropout. Does the model overfit?

---
## 📖 Next Steps
➡️ **Next module:** [02_CNNs/01_convolution_fundamentals.ipynb](../02_CNNs/01_convolution_fundamentals.ipynb) — Learn Convolutional Neural Networks, the backbone of ADAS vision!